# FarmTech Solutions · Fase 6 · **Ir Além 3.1**
## Coleta de imagens por câmera Wi-Fi e reconhecimento em tempo real com o `best.pt`

**Aluno:** Jefferson Gonçalves Lemos · RM 572399

| Material | Link |
|---|---|
| Vídeo de demonstração (até 5 min, não listado) | ⏳ `https://youtu.be/<inserir>` |
| Vídeo complementar sobre a escolha do Hailo | ⏳ `https://youtu.be/<inserir>` |
| Notebook principal (Entregas 1 e 2) | `../JeffersonGoncalvesLemos_rm572399_pbl_fase6.ipynb` |

Neste notebook documento a opção 3.1 do Ir Além. Levei o módulo de visão do **rover autônomo de pasto**
da FarmTech (classes `vaca`, `cobra`, `escorpiao`, `arvore`, `cerca` e `caneca`) para fora do Colab: uma câmera
coleta as imagens e as transmite por Wi-Fi, e um programa Python executado no VS Code reconhece os
objetos com o modelo que treinei na Entrega 1. A `caneca` foi acrescentada como sexta classe por ser o
objeto que tenho à mão para a demonstração ao vivo (seção 5.4); ela não dispara alerta.

## 1. O que o enunciado pede e como atendi

A opção 3.1 do enunciado aceita uma **ESP32-CAM real** ou a **webcam do PC conectada ao Python lendo o `best.pt`**,
e descreve o fluxo: *coletar imagens em tempo real e transmitir para um Visual Code que reconheça os objetos*.

Organizei a solução em **dois modos que usam o mesmo cliente**:

- **BASE (o fluxo descrito no enunciado):** a câmera (ESP32-CAM, Raspberry Pi 5, celular pelo navegador ou webcam) coleta as
  imagens e as transmite como MJPEG por Wi-Fi; o `cliente_vscode.py`, executado no VS Code do PC,
  carrega o `best.pt` e detecta os objetos. A webcam do PC também funciona como fonte local (`--fonte 0`).
- **PLUS (extensão do que foi pedido):** a inferência sai do PC e passa para dentro do rover. O
  Raspberry Pi 5 com a NPU **Hailo-8L** executa o mesmo modelo, compilado em `.hef`, e envia por Wi-Fi o
  vídeo já anotado e as detecções em JSON. Como plano B, o mesmo modelo roda em ONNX Runtime na CPU do Pi.

A tabela relaciona cada requisito do enunciado com a parte do projeto que o atende.

| Requisito do enunciado | Onde está |
|---|---|
| Coleta e envio das imagens | `camera_server.py` (Pi/PC e, com `--fonte navegador`, o celular pela página `celular.html`), `esp32cam/esp32cam_stream.ino` (ESP32-CAM) → `cliente_vscode.py` |
| Integração Wi-Fi estável e eficiente | MJPEG sobre HTTP, reconexão automática, RTT e latência medidos (seções 6 e 7) |
| Escolha e justificativa do hardware | Seção 2 |
| Código organizado e comentado | Pasta `ir_alem/`, seção 4 |
| Figura da arquitetura | Seção 3 (`figuras/arquitetura.png`, de autoria própria) |
| Evidências e vídeo | Seções 7 e 8 |

## 2. Justificativa das escolhas

### Câmera + Wi-Fi em vez de processar tudo no PC com um cabo
O rover se desloca pelo pasto, então a câmera precisa estar **nele**, longe do computador. Escolhi o
Wi-Fi porque é o enlace que já existe na sede da fazenda e não exige infraestrutura nova. O custo é uma
latência variável e eventuais quedas de sinal; por isso o cliente reconecta sozinho e registra a latência
de cada quadro.

### MJPEG sobre HTTP
- É o formato nativo da ESP32-CAM: o sensor OV2640 já entrega JPEG e o microcontrolador apenas repassa os bytes.
- Abre em qualquer navegador, no VLC e no OpenCV, o que simplifica a depuração: basta abrir uma URL.
- Cada quadro é independente. Um pacote perdido compromete apenas um quadro, e não um trecho do vídeo,
  como acontece com H.264. Em troca, cada quadro consome mais banda.

### Celular como câmera Wi-Fi na demonstração ao vivo
Para gravar a demonstração, usei o celular como câmera Wi-Fi. A ideia inicial era um app de câmera IP,
mas o iPhone que uso não permite instalar apps. Por isso, o `camera_server.py` passou a servir uma
página (`/celular`) que o Safari abre e que envia os quadros da câmera por HTTPS ao Mac (seção 5.4).
A vantagem é a câmera de boa qualidade, que pode ser levada até o objeto sem cabo e sem app.
Diferentemente dos apps de câmera IP, que não enviam o `X-Timestamp` nem respondem a `/saude`, essa
fonte mantém todas as métricas do log, inclusive a latência fim a fim, com o relógio do celular
corrigido por um offset estimado pela própria página.

### ESP32-CAM × Raspberry Pi 5 × Pi 5 + Hailo-8L

| | ESP32-CAM | Raspberry Pi 5 | Pi 5 + Hailo-8L (plus) |
|---|---|---|---|
| Papel | só câmera + Wi-Fi | câmera + Wi-Fi + Linux | câmera + Wi-Fi + **inferência embarcada** |
| Processamento | microcontrolador de 240 MHz, PSRAM de 4 MB | CPU ARM de 4 núcleos | CPU + NPU de **13 TOPS** |
| Roda o YOLOv5 do projeto? | não (memória insuficiente) | sim, devagar (ONNX na CPU) | sim, em tempo real (INT8) |
| Depende do PC no campo? | sim | sim (modo BASE) | **não**: só as detecções e os alertas trafegam |
| Quando faz sentido | protótipo barato, câmera fixa | câmera com mais controle e a mesma ótica do dataset | rover autônomo operando em campo |

**Decisão:** uso o modo BASE para demonstrar o fluxo pedido no enunciado com qualquer uma das câmeras.
O plus com Hailo trata a etapa seguinte do cenário da FarmTech: o rover reconhecer os objetos no pasto
sem um computador por perto. Escolhi o Hailo-8L por motivos técnicos:

- **Inferência local:** o modelo roda no próprio rover, sem depender do PC nem de conexão com a nuvem,
  e as imagens não precisam sair do dispositivo.
- **Latência:** a NPU executa o YOLOv5 quantizado em INT8 em poucos milissegundos por quadro, o que
  verifico na seção 8 comparando com a CPU do Pi.
- **Desempenho por consumo:** são 13 TOPS com consumo de poucos watts, compatível com a alimentação de
  um equipamento embarcado.
- **Rede:** como a inferência acontece no rover, o Wi-Fi deixa de ser crítico para a detecção; uma
  queda de sinal atrasa o envio dos alertas, mas não interrompe o reconhecimento.

Detalho essa escolha no vídeo complementar sobre a escolha do Hailo, ⏳ `https://youtu.be/<inserir>`.

### Limitações assumidas
- A latência fim a fim usa o relógio da câmera (`X-Timestamp`) e só é confiável quando a câmera e o PC
  estão sincronizados por NTP. Sem essa sincronização, a métrica de referência passa a ser o **RTT**
  medido pelo próprio cliente.
- No celular pelo navegador, a latência fim a fim depende do offset de relógio estimado pela página
  (incerteza de ±RTT/2) e não inclui o atraso interno da câmera do celular, anterior à entrega do quadro
  ao navegador. O certificado autoassinado exige aceitar o aviso do navegador a cada novo IP do Mac.
- Escolhi transportar quadros JPEG por HTTPS por simplicidade e porque cada quadro leva o seu próprio
  `X-Timestamp`, o que permite medir a latência quadro a quadro. Uma evolução possível é o WebRTC, que
  envia vídeo contínuo com codec e consome menos banda, mas exige o mesmo HTTPS e acrescenta
  dependências como o `aiortc`.
- O modelo foi treinado com cerca de 32 imagens por classe. Em vídeo ao vivo, espero detecções que
  oscilam entre quadros, principalmente de `escorpiao`.
- A quantização INT8 do Hailo pode reduzir a precisão nas classes pequenas. Em vez de supor esse efeito,
  eu o meço na seção 8.

## 3. Arquitetura

![Arquitetura do Ir Além](figuras/arquitetura.png)

*Figura de autoria própria (`figuras/arquitetura.svg`). A faixa verde mostra o fluxo pedido no enunciado
e a faixa azul, o plus embarcado. As duas trilhas chegam ao mesmo cliente no VS Code.*

## 4. Componentes

```
ir_alem/
├── camera_server.py          # câmera → Wi-Fi (Pi ou PC); modo PLUS com --inferencia hailo|onnx
├── celular.html              # página /celular: câmera do celular pelo navegador (--fonte navegador)
├── cliente_vscode.py         # PC: recebe, detecta (best.pt ou .onnx), alerta e grava os logs
├── exportar_modelo.py        # best.pt → best.onnx (opset 11, entrada do Hailo DFC) e, opcionalmente, CoreML
├── backends/                 # inferência embarcada com interface comum detect(frame) -> lista
│   ├── __init__.py           #   6 classes, cores e função anotar()
│   ├── yolo_utils.py         #   letterbox, decodificação do YOLOv5 e NMS em numpy
│   ├── onnx_backend.py       #   plano B: ONNX Runtime (CPU, CoreML, CUDA ou DirectML)
│   ├── compat.py             #   carga do best.pt do Colab em outro Python ou sistema operacional
│   └── hailo_backend.py      #   plus: HailoRT na NPU Hailo-8L
├── esp32cam/esp32cam_stream.ino   # ESP32-CAM servindo as mesmas rotas
├── hailo/README_conversao.md      # best.pt → ONNX → .hef
├── figuras/arquitetura.svg / .png
├── certs/                    # certificado autoassinado do --https (gerado, fora do git)
└── logs/                     # frames.csv e alertas.csv gerados pelo cliente
```

### 4.1 Transmissão: o mesmo contrato em todas as câmeras
Defini as mesmas rotas para todas as câmeras, de modo que o cliente não precisa saber qual hardware
está do outro lado:

| Rota | Conteúdo |
|---|---|
| `/stream` | MJPEG contínuo; cada quadro leva `X-Timestamp` (instante da captura) |
| `/frame.jpg` | um quadro |
| `/saude` | JSON de status, usado para medir o RTT |
| `/deteccoes` | JSON com as detecções (só no modo PLUS) |
| `/celular` | página que transforma o navegador do celular em câmera (só com `--fonte navegador`) |
| `/quadro` | `POST` de um JPEG vindo da página, com `X-Timestamp` (ms) e o offset de relógio estimado |
| `/qr.svg` | QR code com a URL de `/celular` |

No trecho abaixo, do `camera_server.py`, cada quadro novo libera todas as conexões abertas, e nenhuma
delas recebe o mesmo quadro duas vezes:
```python
with estado.cond:
    estado.cond.wait_for(lambda: estado.numero != visto, timeout=5)
    jpeg, visto, ts = estado.jpeg, estado.numero, estado.ts
self.wfile.write(f"--{FRONTEIRA}\r\nContent-Type: image/jpeg\r\n"
                 f"Content-Length: {len(jpeg)}\r\nX-Timestamp: {ts:.6f}\r\n\r\n".encode())
```

### 4.2 Recepção e reconhecimento: `cliente_vscode.py`
O cliente interpreta o multipart por conta própria, em vez de usar `cv2.VideoCapture`, para ter acesso
ao `X-Timestamp`. Se o Wi-Fi cair, ele tenta reconectar a cada 2 s sem travar a janela. No modo BASE, o
`best.pt` é carregado pelo hub do YOLOv5; o `trust_repo=True` evita a confirmação interativa que as
versões recentes do PyTorch pedem ao baixar o repositório. A carga fica dentro de
`carregamento_portavel()` (`backends/compat.py`), que contorna as diferenças de versão do Python e de
sistema operacional entre o treino no Colab e o PC (detalhes na seção 5.1):
```python
with carregamento_portavel():
    self.modelo = torch.hub.load("ultralytics/yolov5", "custom", path=str(caminho),
                                 trust_repo=True)
resultado = self.modelo(frame[:, :, ::-1])            # o hub espera RGB
```
Detecções de `cobra` e `escorpiao` acendem uma faixa vermelha de alerta e geram uma linha em
`alertas.csv`. Cada quadro gera uma linha em `frames.csv`, que é a base da análise das seções 6 e 7.

### 4.3 Inferência embarcada: `backends/`
Reimplementei em numpy o pós-processamento do YOLOv5 (letterbox, decodificação e NMS por classe) para
que o Pi execute a inferência **sem PyTorch**. O backend ONNX confere, ao carregar o modelo, se o número
de classes da saída (`5 + nc`) bate com a lista de nomes informada; sem essa checagem, um modelo COCO
lido com os nomes do projeto produziria rótulos errados sem nenhum erro. O backend Hailo usa o NMS do próprio chip quando o `.hef`
é compilado com o config `yolov5s` do Model Zoo (detalhes em `hailo/README_conversao.md`). A opção
`--classes coco` permite validar o pipeline com um `.hef` pré-treinado no COCO enquanto o `best.hef`
do projeto não está pronto.

## 5. Como executar

### 5.1 Instalação por sistema operacional
Desenvolvi e testei no macOS (Apple Silicon), mas o cliente e os scripts usam apenas `pathlib` e o
próprio interpretador Python, sem comandos de shell, e rodam também em Linux e Windows. A diferença
entre os sistemas está na ativação do ambiente virtual e no pacote do ONNX Runtime:

| Sistema | Ambiente e dependências (dentro de `ir_alem/`) |
|---|---|
| macOS (Apple Silicon) | `python3 -m venv .venv` · `source .venv/bin/activate` · `pip install -r requirements-pc.txt` |
| Linux | os mesmos comandos do macOS; com GPU NVIDIA, trocar `onnxruntime` por `onnxruntime-gpu` e instalar o PyTorch com CUDA (seletor de pytorch.org) |
| Windows (PowerShell) | `py -3.12 -m venv .venv` · `.venv\Scripts\Activate.ps1` · `pip install -r requirements-pc.txt`; com GPU, trocar `onnxruntime` por `onnxruntime-directml` (qualquer GPU DirectX 12) ou `onnxruntime-gpu` (NVIDIA) |

Com o ambiente ativo, os comandos abaixo são iguais nos três sistemas (`python ...`). O `best.pt` da
Entrega 1 fica em `ir_alem/`. O `best.onnx`, usado pelo plano B, pelo cliente sem PyTorch e pelo
Hailo DFC, é gerado por:
```
python exportar_modelo.py --pesos best.pt                       # best.onnx (opset 11, 640 px)
python exportar_modelo.py --pesos best.pt --formato ambos       # no Mac, também best.mlpackage (CoreML)
python exportar_modelo.py --pesos best.pt --so-portavel         # só a cópia best_portavel.pt
```
Antes de exportar, o script grava ao lado dos pesos a cópia `best_portavel.pt`, e é ela que segue para
o `export.py`. Essa etapa existe porque treinei no Colab com Python 3.13 em Linux, e o checkpoint do
YOLOv5 guarda um objeto `pathlib` entre os argumentos do treino: em Python 3.12 ele aponta para o módulo
`pathlib._local`, que não existe nessa versão, e no Windows ele é um `PosixPath`, que o sistema não
consegue instanciar. Na cópia, esses objetos foram convertidos para texto, e ela carrega em qualquer
versão do Python e em qualquer sistema operacional; é o arquivo que distribuo para quem for executar
o projeto. O `cliente_vscode.py` aplica o mesmo ajuste em memória ao abrir o `best.pt` original, então
os dois arquivos funcionam no modo BASE.

### 5.2 Onde a inferência roda: `--dispositivo` e `--provedor`
Deixei o hardware de inferência parametrizável, porque o mesmo cliente precisa aproveitar a GPU do Mac
em que desenvolvi e as GPUs de quem for executar em Windows ou Linux:

| Parâmetro | Vale para | Valores | O que `auto` escolhe |
|---|---|---|---|
| `--dispositivo` | `best.pt` (PyTorch, `cliente_vscode.py`) | `auto`, `cpu`, `mps`, `cuda` | CUDA > MPS > CPU |
| `--provedor` | `best.onnx` (ONNX Runtime, `cliente_vscode.py` e `camera_server.py`) | `auto`, `cpu`, `coreml`, `cuda`, `directml` | CUDA > CoreML > DirectML > CPU |

| Sistema | `best.pt` | `best.onnx` |
|---|---|---|
| macOS (Apple Silicon) | `--dispositivo mps` | `--provedor coreml` (já incluso no `onnxruntime`) |
| Linux ou Windows com NVIDIA | `--dispositivo cuda` | `--provedor cuda` (`onnxruntime-gpu`) |
| Windows sem NVIDIA | `--dispositivo cpu` | `--provedor directml` (`onnxruntime-directml`) |
| Qualquer sistema sem GPU, Raspberry Pi | `--dispositivo cpu` | `--provedor cpu` |

No `onnxruntime`, a CPU fica sempre como reserva para os operadores que o acelerador não suporta. Um
provedor pedido explicitamente e não instalado gera erro com a lista dos disponíveis, em vez de cair
na CPU sem aviso. O cliente imprime no início o que foi efetivamente usado (por exemplo,
`execução onnx:CoreMLExecutionProvider` ou `execução pt:mps`) e grava o mesmo valor na coluna
`execucao` do `frames.csv`. No modo PLUS, o valor vem do `/saude` do servidor (por exemplo,
`remoto:onnx:CPUExecutionProvider`). A análise da seção 6 separa os resultados por essa coluna.

### 5.3 Cenários

| Cenário | Câmera | PC (terminal do VS Code) |
|---|---|---|
| BASE · webcam do PC | não se aplica | `python cliente_vscode.py --fonte 0 --modelo best.pt` |
| BASE · celular pelo navegador (iPhone ou Android, sem app) | no Mac: `python camera_server.py --fonte navegador`; no celular: abrir `https://<IP do Mac>:8000/celular` | `python cliente_vscode.py --fonte https://<IP do Mac>:8000/stream --inseguro --modelo best.pt` |
| BASE · celular com app de câmera IP (Android) | app de câmera IP no celular, na mesma rede Wi-Fi | `python cliente_vscode.py --fonte http://<IP do celular>:<porta>/<rota> --modelo best.pt` |
| BASE · ESP32-CAM | gravar o `esp32cam_stream.ino` e ler o IP no Monitor Serial | `python cliente_vscode.py --fonte http://<IP>/stream --modelo best.pt` |
| BASE · Raspberry Pi 5 | `python camera_server.py --fonte picamera2` | `python cliente_vscode.py --fonte http://<IP>:8000/stream --modelo best.pt` |
| BASE · `.onnx` no PC, sem PyTorch | qualquer uma das anteriores | `python cliente_vscode.py --fonte <URL> --modelo best.onnx --provedor auto` |
| PLUS · Pi 5 + Hailo-8L | `python camera_server.py --fonte picamera2 --inferencia hailo --modelo best.hef` | `python cliente_vscode.py --fonte http://<IP>:8000/stream` |
| PLUS · plano B (CPU do Pi) | `python camera_server.py --fonte picamera2 --inferencia onnx --modelo best.onnx --provedor cpu` | `python cliente_vscode.py --fonte http://<IP>:8000/stream` |
| Teste do pipeline sem `best.hef` | `python camera_server.py --fonte teste.mp4 --loop --inferencia hailo --modelo /usr/share/hailo-models/yolov8s_h8l.hef --classes coco` | `python cliente_vscode.py --fonte http://<IP>:8000/stream` |

Na janela, a tecla **s** salva o quadro atual em `prints/` e a tecla **q** encerra. Sem tela (SSH ou
testes), o cliente roda com `--sem-janela --max-frames 300`.

### 5.4 Demonstração ao vivo: celular como câmera Wi-Fi pelo navegador e a caneca
Na demonstração ao vivo, o celular faz o papel da câmera do rover. Como o iPhone que uso não permite
instalar um app de câmera IP, acrescentei ao `camera_server.py` a fonte `navegador`: o próprio servidor,
rodando no Mac, entrega a página `/celular` (`celular.html`, sem bibliotecas externas), que captura a
câmera traseira com `getUserMedia` e envia cada quadro em JPEG por `POST /quadro` pela rede Wi-Fi. A
partir daí, o caminho é o mesmo das outras câmeras: os quadros alimentam o `/stream`, e o
`cliente_vscode.py` os reconhece com o `best.pt`. A página funciona no Safari do iPhone e no Chrome do
Android, sem instalar nada.

Os navegadores só liberam a câmera em contexto seguro (HTTPS), por isso essa fonte liga o `--https`
por padrão. Na primeira execução, o servidor gera um certificado autoassinado em `ir_alem/certs/`
(fora do git), com o IP do Mac nas SANs, usando o pacote `cryptography` ou, na falta dele, o `openssl`.

Passo a passo:
1. Conectar o Mac e o celular à mesma rede Wi-Fi.
2. No Mac, iniciar o servidor com `python camera_server.py --fonte navegador`. O terminal mostra a URL
   `https://<IP do Mac>:8000/celular` e um QR code com ela; o mesmo QR code fica em
   `https://localhost:8000/qr.svg`. Sem o pacote `segno`, o servidor funciona igual e mostra só a URL.
3. No celular, apontar a câmera nativa para o QR code e tocar no link (ou digitar a URL no navegador).
4. Na primeira visita, o navegador avisa que a conexão não é privada, porque o certificado é
   autoassinado. No Safari: **Mostrar Detalhes** → **visitar este site** → **Visitar Site**. No Chrome
   do Android: **Avançado** → **Ir para <IP> (não seguro)**. O aviso volta se o IP do Mac mudar.
5. Tocar em **Iniciar câmera** e permitir o acesso. O iOS exige esse toque: a câmera não pode ser aberta
   automaticamente ao carregar a página. A página mostra a prévia com as caixas detectadas, o FPS
   enviado, o FPS de detecções recebidas, a latência de ida e volta de cada `POST`, o atraso das caixas
   e o offset do relógio. Os padrões são 640×480, 15 quadros/s e JPEG com qualidade 0,7, ajustáveis na
   tela (até 30 quadros/s). Enquanto a captura está ligada, a página pede ao sistema que mantenha a tela
   acesa (Screen Wake Lock, Safari 16.4 ou posterior) e reabre a câmera quando volta a ficar visível.
6. No terminal do VS Code, iniciar o cliente. A opção `--inseguro` aceita o certificado autoassinado
   apenas nas conexões com esse servidor (stream e `/saude`):
```
python cliente_vscode.py --fonte https://<IP do Mac>:8000/stream --inseguro --modelo best.pt --dispositivo auto
```
O envio é sequencial: a página só manda o próximo quadro depois da resposta do anterior. Assim, se o
Wi-Fi ficar lento, o FPS cai, mas não se forma fila e a latência não cresce. No servidor, guardo apenas
o quadro mais recente pelo mesmo motivo. Com `--inferencia onnx` ou `hailo`, o celular também alimenta o
modo PLUS.

**Caixas na tela do celular.** Na demonstração, quem segura o celular precisa ver o que o modelo
detectou sem olhar para o Mac. Em vez de devolver ao celular o vídeo já anotado, optei por devolver só
as coordenadas das caixas. A prévia continua sendo o `<video>` local, alimentado direto pela câmera, e
um `<canvas>` transparente por cima desenha as caixas. As coordenadas são normalizadas (`[x1, y1, x2, y2]`
entre 0 e 1, relativas ao quadro enviado) e acompanham a classe, a confiança e a identificação do quadro
analisado (número e `X-Timestamp` original). Escolhi esse desenho por três motivos:

- **Banda:** cada atualização tem algumas centenas de bytes de JSON, contra dezenas de kB de um JPEG
  anotado, e o caminho de volta não disputa o Wi-Fi com o envio dos quadros.
- **Fluidez:** a prévia nativa roda a 30 quadros/s com latência praticamente nula, independentemente do
  FPS de envio e do tempo de inferência. Um vídeo anotado de volta mostraria a cena com o atraso da ida,
  da inferência e da volta.
- **Independência da tela:** como as coordenadas são frações do quadro, a página as converte para a área
  em que o vídeo de fato aparece, qualquer que seja o tamanho da tela, a orientação ou o `devicePixelRatio`.
  O vídeo usa `object-fit: contain` (o quadro inteiro fica visível, como o modelo o viu), e a conversão
  desconta as faixas laterais ou superiores que sobram quando as proporções diferem.

As detecções chegam de carona na resposta do próprio `POST /quadro`, sem requisição extra. No modo BASE,
o `cliente_vscode.py` descobre pelo `/saude` que a fonte é o navegador e devolve as detecções de cada
quadro ao servidor (`POST /deteccoes`), em thread própria e sempre só o resultado mais recente, para não
atrasar a inferência. No modo PLUS, o servidor usa as próprias detecções. As cores das classes vêm do
`/saude` e são as mesmas de `backends/__init__.py`; `cobra` e `escorpiao` ganham contorno reforçado, uma
faixa vermelha de alerta e a borda pulsante na prévia. Como a página sabe o instante de captura do quadro
a que as caixas se referem, ela descarta caixas com mais de 500 ms, para que não fiquem "grudadas" na
tela quando as detecções param de chegar, e interpola a posição entre duas atualizações. A limitação é
que as caixas sempre se referem a um quadro alguns milissegundos mais antigo que a prévia; com o objeto
ou o celular em movimento rápido, a caixa segue o objeto com um pequeno atraso.

**Fluidez e quedas do celular.** No primeiro teste com o iPhone (492 quadros, mediana de 66 ms de
latência fim a fim, RTT do Wi-Fi de 3,5 ms e inferência de 23 ms com o `yolov5s.pt` em MPS), o envio
ficou em 9,9 quadros/s, que era o padrão da página, e não o limite do pipeline; por isso o padrão passou
a 15 quadros/s. O teste também mostrou um problema: quando o celular parava de enviar, o `/stream`
ficava sem quadros, o cliente estourava o timeout de leitura e entrava em ciclos de reconexão, com a
janela congelada. Corrigi nos dois lados. O servidor reenvia o último quadro a cada 0,5 s, marcado com o
cabeçalho `X-Keepalive`, e registra uma única linha quando o celular desconecta e outra quando ele volta.
O cliente ignora os quadros de keepalive, mostra "aguardando quadros" na janela (que continua
respondendo às teclas) e, em uma queda real da conexão, imprime uma só mensagem e reconecta em segundo
plano. A leitura do stream fica em uma thread separada que guarda apenas o JPEG mais recente; se a
inferência estiver ocupada, os quadros intermediários são descartados, em vez de acumular atraso.

Validei essas mudanças no Mac, com o script que simula a página enviando o vídeo de teste a 15 quadros/s
em 640×480 (JPEG 0,7) e lendo as respostas do `POST /quadro`, como a página faz. Usei os modelos
pré-treinados no COCO da seção 5.5, com o cliente em `--max-frames 300`:

| Medida (simulação no próprio Mac) | BASE (`yolov5s.pt`, MPS) | PLUS (`yolov5s.onnx` no servidor, CoreML) |
|---|---|---|
| FPS enviado pelo celular simulado | 14,2 | 14,3 |
| FPS de detecções novas recebidas pelo celular | 14 (mediana por segundo) | 14 (mediana por segundo) |
| Idade das caixas ao chegar ao celular (mediana) | 74 ms | 72 ms |
| FPS no cliente (mediana) | 14,2 | 14,3 |
| Latência fim a fim no cliente (mediana / p95) | 27 / 35 ms | 17 / 21 ms |
| Inferência (mediana) | 23 ms | 13 ms |

Todas as caixas devolvidas ficaram dentro do intervalo 0 a 1, e as vacas do vídeo voltaram como `cow`,
com as caixas normalizadas coincidindo com o animal quando redesenhadas sobre o quadro enviado. A idade
das caixas inclui a espera até a resposta do próximo `POST` (até um intervalo de envio, 67 ms a 15
quadros/s), por isso é maior que a latência fim a fim do cliente. No teste de queda, o celular simulado
parou de enviar por 10 s: o servidor registrou a desconexão e a volta, e o cliente mostrou a espera, não
gerou nenhum timeout e retomou sozinho até completar os 300 quadros. Esses números são do próprio Mac,
sem o trecho de Wi-Fi; com o iPhone, a idade das caixas ganha o RTT da rede e o tempo de compressão do
JPEG no celular.

**Latência e relógio.** O `X-Timestamp` de cada quadro é o horário da captura no relógio do celular, que
pode estar adiantado ou atrasado em relação ao Mac. Ao iniciar, e depois a cada 30 s, a página faz
algumas requisições `GET /saude` e estima o offset pelo método de Cristian: o horário informado pelo
servidor corresponde ao ponto médio entre o envio e a resposta, e fico com a amostra de menor RTT. O
servidor aplica esse offset antes de repassar o quadro ao `/stream` e o exibe no terminal e em `/saude`
(campo `celular`). A incerteza da correção é de ±RTT/2 da melhor amostra. O instante registrado é o do
desenho no canvas, então o atraso interno da câmera do celular fica fora da medida. Validei o protocolo
sem celular, com um script que lê um vídeo e faz `POST /quadro` a 10 quadros/s com o relógio
deliberadamente deslocado em 2,5 s. Pelo mesmo método da página, o script estimou o offset com erro
abaixo de 1 ms, e a latência fim a fim registrada no `frames.csv` ficou positiva e coerente (mediana de
28 ms, no próprio Mac), em vez dos −2,5 s que apareceriam sem a correção.

**Alternativa: Câmera de Continuidade do macOS.** Com o iPhone e o Mac na mesma conta Apple (iOS 16 e
macOS 13 ou posteriores), o iPhone aparece no Mac como uma webcam e pode ser lido pelo OpenCV pelo
índice, por exemplo `python camera_server.py --fonte 1` ou `python cliente_vscode.py --fonte 1 --modelo best.pt`.
O índice depende das câmeras conectadas. A imagem continua chegando sem fio, mas pelo protocolo da
Apple, e não pelo HTTP do projeto; o `X-Timestamp` passa a ser o instante da leitura no Mac, e o trecho
entre o celular e o Mac fica fora da medida de latência. Por isso, mantenho essa opção como plano B.

O objeto da demonstração é uma **caneca**. Como não tenho vaca, cobra ou escorpião reais à disposição
durante a gravação, acrescentei a `caneca` ao dataset como sexta classe, no fim da lista
(`vaca, cobra, escorpiao, arvore, cerca, caneca`), para não renumerar as classes anteriores. Ela não
é uma classe crítica: aparece com a caixa desenhada, mas não acende o alerta. O alerta é demonstrado
com imagens de cobra e escorpião exibidas na tela, como o enunciado permite.

### 5.5 Validação sem câmera e sem o `best.pt`
Antes de ter o `best.pt`, validei o caminho completo com o `yolov5s.pt` pré-treinado no COCO, em que
a vaca aparece como `cow`. Um vídeo faz o papel da câmera, e a opção `--classes coco` informa ao
backend ONNX os 80 nomes do COCO:
```
python exportar_modelo.py --pesos yolov5s.pt
python camera_server.py --fonte teste.mp4 --loop --fps-max 15
python cliente_vscode.py --fonte http://localhost:8000/stream --modelo yolov5s.onnx --classes coco --provedor cpu --sem-janela --max-frames 300
python cliente_vscode.py --fonte http://localhost:8000/stream --modelo yolov5s.onnx --classes coco --provedor coreml --sem-janela --max-frames 300
python cliente_vscode.py --fonte http://localhost:8000/stream --modelo yolov5s.pt --dispositivo cpu --sem-janela --max-frames 300
python cliente_vscode.py --fonte http://localhost:8000/stream --modelo yolov5s.pt --dispositivo mps --sem-janela --max-frames 300
```
O PLUS também pode ser simulado no próprio PC, com o servidor fazendo a inferência em ONNX:
```
python camera_server.py --fonte teste.mp4 --loop --fps-max 15 --inferencia onnx --modelo yolov5s.onnx --classes coco
python cliente_vscode.py --fonte http://localhost:8000/stream --sem-janela --max-frames 300
```
Cada execução acrescenta uma sessão ao `logs/frames.csv`, e as células da seção 6 comparam as
sessões pela coluna `execucao`. Em Windows e Linux, os mesmos comandos valem trocando `coreml`/`mps`
por `directml` ou `cuda`.

## 6. Protocolo de medição

Para comparar os cenários em condições equivalentes, fixei: **a mesma cena** (o pasto, ou a TV exibindo
imagens de cobra e escorpião, como o enunciado permite), **a mesma rede Wi-Fi**, **2 minutos por cenário**
e o cliente gravando em `logs/frames.csv`.

| Métrica | Como é medida | Coluna do log |
|---|---|---|
| FPS fim a fim | quadros exibidos por segundo no PC (janela de 1 s) | `fps` |
| Latência fim a fim | instante de exibição − `X-Timestamp` da captura (exige NTP; no celular pelo navegador, o relógio é corrigido pelo offset da seção 5.4) | `latencia_fim_a_fim_ms` |
| RTT de rede | ida e volta de `GET /saude`, a cada 100 ms | `rtt_rede_ms` |
| Latência de inferência | tempo do `detect()`: no PC (BASE) ou no Pi (PLUS) | `latencia_inferencia_ms` |
| Onde a inferência rodou | `pt:<dispositivo>`, `onnx:<provider>` ou `remoto:<backend>` (seção 5.2) | `execucao` |
| Alertas | quadros com `cobra` ou `escorpiao` | `alerta` e `alertas.csv` |

**Hipóteses que pretendo verificar:**
1. No BASE, o gargalo é a inferência no PC sem GPU, e não o Wi-Fi.
2. No PLUS com Hailo, a inferência cai para poucos milissegundos e o FPS passa a ser limitado pela câmera e pela rede.
3. O plano B (ONNX na CPU do Pi) funciona, mas com FPS baixo demais para um rover em movimento.
4. No PC, o acelerador (MPS ou CoreML no Mac, CUDA ou DirectML em Windows/Linux) reduz a latência de
   inferência em relação à CPU, a ponto de o FPS passar a ser limitado pela câmera.

In [ ]:
# Análise dos logs gravados pelo cliente_vscode.py (executado antes nos cenários da seção 6)
from pathlib import Path                                   # caminhos independentes do sistema operacional
import pandas as pd                                        # leitura e agregação dos logs
import matplotlib.pyplot as plt                            # gráficos da análise

LOGS = Path("logs")                                        # pasta padrão do cliente (opção --logs)
frames_csv, alertas_csv = LOGS / "frames.csv", LOGS / "alertas.csv"

if frames_csv.exists():
    frames = pd.read_csv(frames_csv)                       # uma linha por quadro exibido
    alertas = pd.read_csv(alertas_csv) if alertas_csv.exists() else pd.DataFrame()
    if "execucao" not in frames:                           # logs de versões anteriores do cliente não têm a coluna
        frames["execucao"] = ""
    execucao = frames["execucao"].fillna("").str.replace("ExecutionProvider", "")   # onnx:CoreMLExecutionProvider → onnx:CoreML
    frames["cenario"] = (frames["modo"] + " · " + execucao).str.rstrip(" ·")   # ex.: BASE · pt:mps
    print(f"{len(frames)} quadros em {frames['sessao'].nunique()} sessão(ões); {len(alertas)} alertas")
    print("Cenários:", ", ".join(sorted(frames["cenario"].unique())))   # modo + onde a inferência rodou
    display(frames.tail())
else:
    frames, alertas = None, None
    print("⏳ Ainda não há logs/frames.csv: execute o cliente_vscode.py nos cenários da seção 6.")

In [ ]:
# Resumo por cenário (modo + dispositivo/provedor): mediana e percentil 95 de cada métrica
if frames is not None:
    def p95(serie):                                        # percentil 95: o pior caso que ainda ocorre com frequência
        return serie.quantile(0.95)

    metricas = ["fps", "latencia_fim_a_fim_ms", "rtt_rede_ms", "latencia_inferencia_ms"]
    resumo = (frames[frames["frame"] > 30]                 # descarta o aquecimento (primeiros 30 quadros, cerca de 1 s)
              .groupby("cenario")[metricas]
              .agg(["median", p95])
              .round(1))
    display(resumo)

In [ ]:
# Gráficos: distribuição das latências e FPS ao longo do tempo, por cenário
if frames is not None:
    dados = frames[frames["frame"] > 30].copy()
    dados["t"] = dados.groupby("sessao")["timestamp"].transform(lambda s: s - s.min())   # segundos desde o início da sessão

    fig, eixos = plt.subplots(1, 3, figsize=(16, 4.5))
    modos = sorted(dados["cenario"].unique())                # um cenário por modo + dispositivo/provedor
    for eixo, coluna, titulo in [(eixos[0], "latencia_inferencia_ms", "Inferência (ms)"),
                                 (eixos[1], "latencia_fim_a_fim_ms", "Fim a fim (ms)")]:
        series = [dados.loc[dados["cenario"] == m, coluna].dropna() for m in modos]
        eixo.boxplot(series, showfliers=False)             # caixa = quartis; outliers ocultos
        eixo.set_xticks(range(1, len(modos) + 1), modos, rotation=20, ha="right")   # nomes dos cenários no eixo x
        eixo.set_title(titulo); eixo.grid(axis="y", alpha=0.3)
    for m in modos:                                        # FPS ao longo do tempo, uma linha por cenário
        sub = dados[dados["cenario"] == m]
        eixos[2].plot(sub["t"], sub["fps"], label=m, lw=1)
    eixos[2].set_title("FPS no tempo"); eixos[2].set_xlabel("s"); eixos[2].legend(); eixos[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# Alertas de risco biológico: quantidade, classe e confiança
if alertas is not None and len(alertas):
    display(alertas.groupby("classe")["conf"].agg(["count", "mean", "min", "max"]).round(3))
    alertas["conf"].plot.hist(bins=20, title="Confiança dos alertas", figsize=(6, 3.5))
    plt.xlabel("confiança"); plt.show()
else:
    print("⏳ Ainda não há alertas registrados.")

## 7. Evidências

> ⏳ Pendente de execução: após rodar cada cenário da tabela da seção 5, insiro aqui um print salvo com
> a tecla **s** (`prints/`) e comento o que ele mostra (caixas detectadas, FPS e latências exibidas na janela).

| Cenário | Print | Observação |
|---|---|---|
| BASE · webcam do PC | ⏳ `prints/...jpg` | |
| BASE · celular pelo navegador (caneca ao vivo) | ⏳ `prints/...jpg` | |
| BASE · ESP32-CAM | ⏳ `prints/...jpg` | |
| BASE · Raspberry Pi 5 | ⏳ `prints/...jpg` | |
| PLUS · Hailo-8L | ⏳ `prints/...jpg` | |
| Alerta de cobra (TV) | ⏳ `prints/...jpg` | |

No vídeo de demonstração, também abro a rota `/saude` no navegador do celular, o que comprova que o
dispositivo está acessível pela rede Wi-Fi.

## 8. Plus: Raspberry Pi 5 + Hailo-8L

A conversão `best.pt → ONNX → .hef` está descrita em `hailo/README_conversao.md`; o ONNX de entrada do
DFC é o mesmo gerado pelo `exportar_modelo.py` (seção 5.1).

Comparo o **benchmark só da NPU** (`hailortcli benchmark best.hef`) com as medições **fim a fim** (logs da seção 6):

| | FPS | Latência de inferência (ms) | Observação |
|---|---|---|---|
| `hailortcli benchmark` (só NPU) | ⏳ | ⏳ | limite teórico do chip |
| PLUS Hailo, fim a fim | ⏳ | ⏳ | + captura + JPEG + Wi-Fi |
| PLUS ONNX na CPU do Pi | ⏳ | ⏳ | plano B |
| BASE no PC (`best.pt`) | ⏳ | ⏳ | referência |

> ⏳ Pendente de execução: com o benchmark e as sessões do cliente, comparo o FPS do chip isolado com o
> FPS fim a fim. A diferença indica quanto do tempo vai para captura, codificação JPEG e Wi-Fi.

**Efeito da quantização INT8** (mesmo conjunto de teste da Entrega 1):

| Classe | mAP@0.5 FP32 (Colab) | mAP@0.5 INT8 (Hailo) | Δ |
|---|---|---|---|
| vaca | ⏳ | ⏳ | |
| cobra | ⏳ | ⏳ | |
| escorpiao | ⏳ | ⏳ | |
| arvore | ⏳ | ⏳ | |
| cerca | ⏳ | ⏳ | |
| caneca | ⏳ | ⏳ | |

> Hipótese: o INT8 penaliza mais as classes de objeto pequeno (`escorpiao`, `cobra`). Se isso se
> confirmar, o rover vai precisar de mais resolução de entrada ou de um segundo estágio de classificação
> para as classes de segurança.

## 9. Conclusões

> ⏳ Pendente de execução: completo esta seção com os números medidos nas seções 6 a 8.

- **Funcionalidade:** a câmera coleta as imagens, o Wi-Fi as transmite e o VS Code reconhece os objetos
  com o `best.pt` da Entrega 1, com os quatro tipos de câmera (incluindo o celular) e o mesmo cliente,
  em macOS, Linux ou Windows, escolhendo o hardware de inferência por `--dispositivo` e `--provedor`.
- **Wi-Fi:** ⏳ RTT mediano de __ ms e latência fim a fim p95 de __ ms; avalio também se o cliente se
  recuperou sozinho das quedas de sinal.
- **Plus:** ⏳ com o Hailo-8L, a inferência passou de __ ms (CPU do Pi) para __ ms, e o rover deixa de
  depender do PC no campo.
- **Limitações:** dataset pequeno (cerca de 32 imagens de treino por classe), latência fim a fim
  dependente de NTP, possível perda de precisão com INT8 nas classes pequenas e testes de cobra e
  escorpião feitos com imagens na TV, não com animais reais.